# Step 1b.4 — Passage Encoding & FAISS Indexing

**Where we are:** Step 1 — Corpus Preparation (final part)  
**What we do:** Encode all ~42M sentence-aligned passages with Contriever, then build sharded FAISS indices  
**How it connects:** The FAISS index is used in Step 4 (baseline retrieval) and Step 5 (KG-enhanced reranking)

### Key concept: FAISS stores only vectors, not text

FAISS is a pure similarity engine — it stores **only numerical vectors** (768 floats per passage), with no text or metadata.  
When we search, FAISS returns the **position** of each matching vector inside the index (0, 1, 2, ...), not the passage itself.

To recover the original text, we need a two-step lookup:

```
FAISS index position → shard_XX_ids.npy → passage ID → TSV corpus → title + text
```

This is why we save a separate `shard_XX_ids.npy` file alongside each shard: it maps each vector position back to the original passage ID in the corpus.

### Architecture

| Component | Detail                                                                                      |
|-----------|---------------------------------------------------------------------------------------------|
| Model | `facebook/contriever` (768-dim, mean pooling)                                               |
| Index type | `IndexFlatIP` — exact inner product (equivalent to cosine similarity on normalized vectors) |
| Precision | float32 (same as Silvestri et al.)                                                          |
| Sharding | ~5M vectors/shard (~15.3 GB each), 9 shards total                                           |
| Search strategy | Sequential per-shard GPU search, merge top-k                                                |

### Why not a single index?
42M × 768 × 4 bytes ≈ 129 GB — exceeds both GPU VRAM (16 GB) and system RAM (64 GB).  
Sharding lets us load one shard at a time on GPU for fast exact search.

## 1. Setup

In [ ]:
import numpy as np
import os
import sys
import math
import time
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm  # progress bars for loops

# WARNING — PERSONAL CONFIG: i tre path qui sotto sono hardcoded sulla
# directory miniconda della macchina dev (C:/Users/Utente/miniconda3/).
# Se installi conda in un altro prefisso o lavori da un'altra macchina,
# MODIFICA i path qui sotto. Necessario perché faiss-gpu su Windows è
# disponibile solo via conda (no pip wheel) — questa workaround linka le
# DLL e importa il package conda dal venv uv.
#
# faiss-gpu lives in the conda environment;
# we add its DLL paths so the native extension can find CUDA/MKL libs
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/bin')
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/lib')
# make conda's faiss package importable from this venv
sys.path.insert(0, 'C:/Users/Utente/miniconda3/Lib/site-packages')

import faiss  # Facebook AI Similarity Search — efficient vector similarity

# Sanity check: verify GPU is available for both FAISS and PyTorch
print(f"faiss {faiss.__version__} — GPU devices: {faiss.get_num_gpus()}")
print(f"torch {torch.__version__} — CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# --- Paths ---
CORPUS_PATH = Path("data/wikipedia_2018_sentence_aligned/psgs_w100_sentence.tsv")
OUTPUT_DIR  = Path("data/faiss_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create output dir if it doesn't exist

# --- Model ---
MODEL_NAME = "facebook/contriever"  # dense retriever, outputs 768-dim embeddings
MAX_LENGTH = 512                     # max tokens per passage (Contriever/BERT limit)

# --- Encoding ---
BATCH_SIZE = 512                     # how many passages we encode in one GPU forward pass
SHARD_SIZE = 5_000_000               # passages per shard (~15.3 GB in float32)
# why shard? ~42M × 768 × 4 bytes ≈ 129 GB — doesn't fit in GPU or RAM at once
# total shards: 9 (shard_00 .. shard_08), last shard has ~2M passages

# --- Multi-machine parallelism ---
# Each shard is fully independent: same model, same IDs, no cross-shard state.
# To split work across machines, set a different range on each one:
#   Machine A: SHARD_START=0, SHARD_END=5
#   Machine B: SHARD_START=5, SHARD_END=9
# Then copy all shard_XX.npy / shard_XX_ids.npy into the same OUTPUT_DIR.
# Resume support (skip existing files) acts as a safety net against overlaps.
# SHARD_START = 0       # first shard to process (inclusive)
# SHARD_END   = None    # last shard to process (exclusive), None = all shards
SHARD_START = 0       # penultimate shard (inclusive)
SHARD_END   = 9       # last shard + 1 (exclusive) — processes shard 7 and 8

# --- Device ---
# torch.device selects GPU if available, otherwise falls back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 3. Load Contriever

Contriever is an unsupervised dense retriever based on BERT.  
It uses **mean pooling** over the last hidden state (not CLS token) to produce 768-dim embeddings.

In [ ]:
# AutoTokenizer.from_pretrained: downloads the tokenizer for the model
# What it loads: vocabulary (30,522 subword tokens for BERT), tokenization rules
# (WordPiece splitting), and special tokens ([CLS], [SEP], [PAD]).
# The "Auto" prefix reads config.json from the HuggingFace repo and picks the
# right class automatically — here it resolves to BertTokenizer.
# Input: raw text string → Output: list of integer token IDs
#   e.g. "George Washington" → [101, 2577, 2899, 102]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# AutoModel.from_pretrained: downloads architecture AND weights of the neural model
# What it loads:
#   - architecture (from config.json): 12 Transformer layers, attention heads, dims
#   - pre-trained weights (from pytorch_model.bin): ~440 MB of float32 parameters
# Again, "Auto" reads config.json → sees "model_type": "bert" → instantiates BertModel.
# You could write BertModel.from_pretrained(...) directly, but Auto decouples your
# code from knowing in advance which architecture the checkpoint uses.
# Input: token IDs (B, S) → Output: hidden states (B, S, 768) — one vector per token
# .to(DEVICE) moves all parameters to GPU memory for fast inference
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# model.eval() switches to inference mode:
# - disables dropout (random neuron deactivation used only during training)
# - BatchNorm layers use stored statistics instead of batch statistics
# this ensures deterministic, reproducible outputs
model.eval()

print(f"Model loaded on {DEVICE}")
print(f"Embedding dim: {model.config.hidden_size}")

## 4. Encoding Utilities

Mean pooling masks out padding tokens before averaging, following the original Contriever implementation.  

**Tensor shapes walkthrough** (batch_size=B, seq_len=S, hidden=768):
- `last_hidden_state`: (B, S, 768) — one vector per token
- `attention_mask`: (B, S) — 1 for real tokens, 0 for padding
- After masking: padding positions become 0-vectors
- `sum(dim=1)`: (B, 768) — sum over sequence
- Divide by token count → (B, 768) — mean embedding per passage

In [ ]:
def mean_pooling(model_output, attention_mask):
    """Contriever-style mean pooling: average non-padding tokens.

    The model outputs one 768-dim vector per token. We need one vector per passage.
    Mean pooling averages all real-token vectors, ignoring padding positions.
    """
    # last_hidden_state shape: (B, S, 768)
    #   B = batch size (number of passages in this batch)
    #   S = sequence length in tokens (subwords, not words — WordPiece may split
    #       one word into multiple tokens, e.g. "Washington" → ["Wash", "##ington"])
    #       S includes special tokens [CLS], [SEP] and [PAD] padding
    #   768 = Contriever's hidden dimension — one vector per token
    last_hidden = model_output.last_hidden_state

    # attention_mask shape: (B, S) — for each sequence in the batch, for each token:
    #   1 = real token, 0 = padding
    #   e.g. with B=2, S=5:
    #   [[1, 1, 1, 0, 0],    ← sequence 1: 3 real tokens, 2 padding
    #    [1, 1, 1, 1, 0]]    ← sequence 2: 4 real tokens, 1 padding
    #
    # unsqueeze(-1): (B, S) → (B, S, 1) — adds a trailing dimension so the mask
    #   can broadcast across all 768 values of each token vector
    #   e.g. [1, 1, 0] → [[1], [1], [0]]
    #
    # ~bool inverts: True where padding → masked_fill sets those positions to 0.0
    #   real token:    [0.23, -0.41, ..., 0.17] → unchanged
    #   padding token: [0.08,  0.55, ..., -0.3] → [0.0, 0.0, ..., 0.0]
    # this way padding tokens contribute nothing to the sum below
    last_hidden = last_hidden.masked_fill(
        ~attention_mask.unsqueeze(-1).bool(), 0.0
    )

    # sum(dim=1): sum token vectors along the sequence axis → (B, 768)
    # divide by the count of real tokens per passage to get the mean
    # keepdim=True keeps shape (B, 1) so division broadcasts correctly across 768 dims
    emb = last_hidden.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)
    return emb  # (B, 768) — one embedding per passage


@torch.no_grad()  # disable gradient computation — saves memory and speeds up inference
def encode_batch(texts, model, tokenizer, device, max_length=MAX_LENGTH):
    """Encode a list of strings into float32 numpy embeddings.

    Pipeline: text → tokenize → GPU forward pass → mean pool → CPU numpy array
    """
    # tokenizer() converts strings to token IDs + attention masks
    # padding=True: pad shorter sequences to match the longest in the batch
    # truncation=True: cut sequences longer than max_length
    # return_tensors="pt": return PyTorch tensors (not lists)
    # .to(device): move input tensors to GPU
    inputs = tokenizer(
        texts,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    # model(**inputs) runs the forward pass on GPU
    # **inputs unpacks the dict as keyword arguments (input_ids=..., attention_mask=...)
    output = model(**inputs)

    # mean pool the token-level outputs into one embedding per passage
    emb = mean_pooling(output, inputs["attention_mask"])

    # .cpu() moves the result from GPU → CPU RAM (required for numpy conversion)
    # .numpy() converts the PyTorch tensor to a numpy array for FAISS/disk storage
    return emb.cpu().numpy()  # (B, 768) float32

## 5. Load Corpus

The TSV has columns `id`, `text`, `title` (~42M rows, 14.5 GB).  
We load it with Polars into memory (~13 GB in Arrow format).  
Following Silvestri et al., each passage is encoded as `title + " " + text`.

In [ ]:
import polars as pl  # fast DataFrame library (alternative to pandas)

print(f"Loading corpus from {CORPUS_PATH} ...")
t0 = time.time()

# pl.read_csv: loads the entire TSV file into memory as a Polars DataFrame
# separator="\t": tab-separated values (TSV format)
# quote_char='"': handle quoted fields containing tabs/newlines
# schema_overrides: force column types (avoids auto-detection issues on 14.5 GB file)
df = pl.read_csv(
    CORPUS_PATH,
    separator="\t",
    quote_char='"',
    has_header=True,
    schema_overrides={"id": pl.Int64, "text": pl.String, "title": pl.String},
)

n_passages = len(df)
# math.ceil rounds up: e.g. 42M / 5M = 8.4, we need 9 shards (last one smaller, ~2M)
n_shards = math.ceil(n_passages / SHARD_SIZE)

print(f"Loaded {n_passages:,} passages in {time.time() - t0:.1f}s")
print(f"Shards: {n_shards} (of {SHARD_SIZE:,} passages each)")
df.head(3)

## 6. Encode Passages into Shards

For each shard:
1. Slice the DataFrame → extract `title + " " + text`
2. Encode in batches of 512 on GPU
3. Save embeddings as `shard_XX.npy` (float32) and IDs as `shard_XX_ids.npy`
4. Validate: correct shape, no NaN, norms ≈ 1.0

**Resume support:** existing shards are skipped automatically.

In [ ]:
def validate_shard(embeddings, shard_idx):
    """Quick sanity checks on a shard's embeddings.

    Catches common issues: wrong dimensions, NaN from numerical errors,
    and unexpected norms (Contriever embeddings should have norms around 1.0).
    """
    # check shape: must be 2D with 768 columns (one per embedding dimension)
    assert embeddings.ndim == 2 and embeddings.shape[1] == 768, \
        f"Shard {shard_idx}: unexpected shape {embeddings.shape}"

    # check for NaN: can occur from division by zero in mean pooling
    # (e.g., if a passage has 0 real tokens — should never happen)
    assert not np.isnan(embeddings).any(), \
        f"Shard {shard_idx}: NaN detected"

    # spot-check norms on first 100 vectors
    # np.linalg.norm computes the L2 norm (length) of each vector
    # Contriever embeddings are roughly unit-norm (~0.9-1.1)
    norms = np.linalg.norm(embeddings[:100], axis=1)
    print(f"  Validation OK — shape {embeddings.shape}, "
          f"norm range [{norms.min():.3f}, {norms.max():.3f}]")

In [ ]:
total_start = time.time()

# resolve shard range: None means "all shards"
shard_end = SHARD_END if SHARD_END is not None else n_shards
print(f"Processing shards [{SHARD_START} .. {shard_end - 1}] out of {n_shards} total\n")

for shard_idx in range(SHARD_START, shard_end):
    # output file paths: embeddings (.npy) and passage IDs (.npy)
    shard_path = OUTPUT_DIR / f"shard_{shard_idx:02d}.npy"
    ids_path   = OUTPUT_DIR / f"shard_{shard_idx:02d}_ids.npy"

    # --- Resume support: skip shards that were already encoded ---
    # useful if the process crashes mid-way (e.g., OOM, power loss)
    if shard_path.exists() and ids_path.exists():
        print(f"Shard {shard_idx}/{n_shards-1} already exists, skipping")
        continue

    # compute the row range for this shard
    start = shard_idx * SHARD_SIZE
    end   = min(start + SHARD_SIZE, n_passages)  # last shard may be smaller
    shard_len = end - start
    print(f"\nShard {shard_idx}/{n_shards-1} — passages [{start:,} .. {end-1:,}] ({shard_len:,})")

    # --- Extract texts for this shard only ---
    # df.slice(offset, length): zero-copy slice of the DataFrame (memory-friendly)
    shard_df = df.slice(start, shard_len)
    titles = shard_df["title"].to_list()   # list of title strings
    texts  = shard_df["text"].to_list()     # list of passage text strings
    shard_ids = shard_df["id"].to_numpy()   # passage IDs as numpy array

    # Concatenate title + text for each passage (same format as Silvestri et al.)
    # zip pairs titles[i] with texts[i]; the f-string joins them with a space
    # if title is empty/None, use text alone to avoid a leading space
    shard_texts = [
        f"{t} {x}" if t else x
        for t, x in zip(titles, texts)
    ]
    del titles, texts, shard_df  # free memory before encoding starts

    # --- Encode in batches on GPU ---
    embeddings_list = []  # accumulate numpy arrays, one per batch
    shard_start = time.time()

    # tqdm wraps the range to display a progress bar
    for batch_start in tqdm(
        range(0, shard_len, BATCH_SIZE),
        desc=f"Shard {shard_idx}",
        total=math.ceil(shard_len / BATCH_SIZE),  # total number of batches
    ):
        # slice out this batch's texts
        batch_texts = shard_texts[batch_start : batch_start + BATCH_SIZE]
        # encode on GPU → get numpy array (B, 768) on CPU
        emb = encode_batch(batch_texts, model, tokenizer, DEVICE)
        embeddings_list.append(emb)

    del shard_texts  # free the text list

    # --- Concatenate all batch results into one array and save ---
    # np.concatenate joins list of (B, 768) arrays → (shard_len, 768)
    shard_embeddings = np.concatenate(embeddings_list, axis=0)
    del embeddings_list

    validate_shard(shard_embeddings, shard_idx)

    # np.save: write array to disk in .npy format (binary, preserves dtype/shape)
    np.save(shard_path, shard_embeddings)
    np.save(ids_path, shard_ids)

    elapsed = time.time() - shard_start
    print(f"  Saved {shard_path.name} ({shard_embeddings.nbytes / 1e9:.1f} GB) in {elapsed/60:.1f} min")

    # free memory and clear GPU cache before next shard
    del shard_embeddings, shard_ids
    torch.cuda.empty_cache()  # releases unused GPU memory back to CUDA

total_elapsed = time.time() - total_start
print(f"\nEncoding complete — total time: {total_elapsed/3600:.1f} hours")

## 7. Build FAISS Indices

One `IndexFlatIP` per shard. Inner product on normalized vectors = cosine similarity.  
Each `.faiss` file can be loaded to GPU at search time independently.

In [ ]:
# use the same shard range defined in Configuration
shard_end = SHARD_END if SHARD_END is not None else n_shards

for shard_idx in range(SHARD_START, shard_end):
    shard_path = OUTPUT_DIR / f"shard_{shard_idx:02d}.npy"
    index_path = OUTPUT_DIR / f"shard_{shard_idx:02d}.faiss"

    # skip already-built indices (resume support)
    if index_path.exists():
        print(f"Index {shard_idx}/{n_shards-1} already exists, skipping")
        continue

    print(f"Building index for shard {shard_idx} ...", end=" ")

    # np.load: read the .npy file back into a numpy array
    embeddings = np.load(shard_path)

    # IndexFlatIP: brute-force index using Inner Product similarity
    # "Flat" = no compression or approximation — exact search
    # "IP" = Inner Product (on normalized vectors, equivalent to cosine similarity)
    # the argument (768) is the vector dimensionality
    index = faiss.IndexFlatIP(embeddings.shape[1])

    # index.add: inserts all vectors into the index
    # after this, the index holds a copy of the data and is ready for search
    index.add(embeddings)

    # faiss.write_index: serialize the index to a .faiss file on disk
    # can be loaded later with faiss.read_index for search
    faiss.write_index(index, str(index_path))
    print(f"done — {index.ntotal:,} vectors")

    del embeddings, index  # free memory

print("\nAll FAISS indices built.")

## 8. Validation — Sanity Check

Quick test: encode a sample query with Contriever, search against one shard on GPU,  
and verify the returned passages make sense.

In [ ]:
# faiss.read_index: load a .faiss file from disk back into memory (CPU)
test_index = faiss.read_index(str(OUTPUT_DIR / "shard_00.faiss"))
test_ids   = np.load(OUTPUT_DIR / "shard_00_ids.npy")

# StandardGpuResources: allocates GPU memory (scratch space, temp buffers) for FAISS
gpu_res = faiss.StandardGpuResources()

# index_cpu_to_gpu: copies the CPU index to GPU for much faster search
# args: (gpu_resources, gpu_device_id, cpu_index)
test_index_gpu = faiss.index_cpu_to_gpu(gpu_res, 0, test_index)

print(f"Shard 0 on GPU: {test_index_gpu.ntotal:,} vectors")

In [ ]:
# Encode a test query with the same model used for passages
# (query and passages must share the same embedding space for similarity to work)
test_query = "who was the first president of the united states"
query_emb = encode_batch([test_query], model, tokenizer, DEVICE)  # (1, 768)

# index.search(query_vectors, k): find the k most similar vectors in the index
# returns:
#   scores: (n_queries, k) — inner product similarity scores, sorted descending
#   indices: (n_queries, k) — positions in the index (not passage IDs!)
k = 5
scores, indices = test_index_gpu.search(query_emb, k)

# Display results: map index positions → passage IDs → original text
print(f"Query: '{test_query}'\n")
print(f"{'Rank':<6} {'Score':<10} {'ID':<12} {'Title':<30} {'Text (first 80 chars)'}")
print("-" * 120)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    # map the FAISS index position to the actual passage ID
    passage_id = test_ids[idx]
    # look up the passage text in the original DataFrame
    row = df.filter(pl.col("id") == passage_id)
    title = row["title"][0] if len(row) > 0 else "?"
    text  = row["text"][0][:80] if len(row) > 0 else "?"
    print(f"{rank:<6} {score:<10.4f} {passage_id:<12} {title:<30} {text}")

In [ ]:
# Cleanup: free GPU memory used by FAISS and PyTorch
del test_index_gpu, test_index, gpu_res
torch.cuda.empty_cache()

# Summary: list all output files with their sizes
print("\nOutput files:")
for f in sorted(OUTPUT_DIR.iterdir()):  # iterdir() lists all files in the directory
    size_mb = f.stat().st_size / 1e6     # file size in megabytes
    print(f"  {f.name:<25} {size_mb:>10,.1f} MB")